In [ ]:
import sys
import time
from tqdm import tqdm

print("=" * 80)
print("INITIALIZING SYSTEM - A100 OPTIMIZED")
print("=" * 80)

print("\n[1/4] Installing dependencies...")
!pip install -q gradio pymupdf pillow pdf2image opencv-python-headless nltk replicate torch
!pip install -q FlagEmbedding
!pip install -q faiss-cpu
!apt-get install -q poppler-utils > /dev/null 2>&1
print("✓ Dependencies installed")

print("\n[2/4] Importing libraries...")
import os
import gradio as gr
from FlagEmbedding import FlagModel, FlagReranker
import fitz
from pdf2image import convert_from_path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
import numpy as np
import faiss
import re
from collections import defaultdict
import logging
from pathlib import Path
from PIL import Image
import replicate
import torch
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

import nltk
nltk.download('punkt', quiet=True)
print("✓ Libraries imported")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

print("\n[3/4] Loading BGE embeddings...")
progress_bge = tqdm(total=100, desc="BGE", bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt}')
embedding_model = FlagModel("BAAI/bge-large-en-v1.5", use_fp16=True)
progress_bge.update(60)
reranker_model = FlagReranker("BAAI/bge-reranker-large", use_fp16=True)
progress_bge.update(40)
progress_bge.close()
print("✓ BGE loaded")

print("\n[4/4] Building system...")

class Config:
    REPLICATE_API_TOKEN = "YOUR_REPLICATE_API_TOKEN"
    CHANDRA_MODEL = "datalab-to/ocr"

    CHUNK_SIZE = 1000
    CHUNK_OVERLAP = 200
    RETRIEVAL_K = 15
    RERANK_TOP_K = 5
    IMAGE_DPI = 300
    MAX_RETRIES = 2
    RETRY_DELAY = 3
    TIMEOUT = 90

os.environ["REPLICATE_API_TOKEN"] = Config.REPLICATE_API_TOKEN

class RegexPatterns:
    CURRENCY = r'\$\s*\d{1,3}(?:,\d{3})*(?:\.\d{2})?'
    PERCENTAGE = r'\d{1,3}(?:\.\d{1,4})?\s*%'
    DATE = r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}'
    LOAN_NUMBER = r'(?:Loan|Account)\s*#?\s*:?\s*([A-Z0-9-]{6,20})'
    INTEREST_RATE = r'(?:Interest Rate|APR)\s*:?\s*(\d{1,2}\.\d{1,4}%?)'

    @classmethod
    def extract_all(cls, text: str) -> Dict:
        patterns = {
            'currency': cls.CURRENCY,
            'percentage': cls.PERCENTAGE,
            'date': cls.DATE,
            'loan_number': cls.LOAN_NUMBER,
            'interest_rate': cls.INTEREST_RATE
        }
        result = {}
        for name, pattern in patterns.items():
            matches = re.findall(pattern, text, re.IGNORECASE)
            if matches:
                result[name] = matches
        return result

@dataclass
class DocumentChunk:
    text: str
    doc_id: str
    chunk_id: int
    page_num: int
    confidence: float
    entities: Dict = field(default_factory=dict)
    embedding: Optional[np.ndarray] = None

@dataclass
class PageData:
    page_num: int
    text: str
    confidence: float
    image_path: str

class ChandraReplicateOCR:
    def __init__(self):
        self.stats = {'attempts': 0, 'successes': 0, 'total_time': 0.0}
        self.executor = ThreadPoolExecutor(max_workers=2)

    def _call_replicate(self, image_path: str) -> str:
        output = replicate.run(
            Config.CHANDRA_MODEL,
            input={
                "file": open(image_path, "rb"),
                "visualize": False,
                "skip_cache": False,
                "return_pages": False
            }
        )

        if isinstance(output, dict):
            return output.get('text', '') or output.get('content', '')
        elif isinstance(output, str):
            return output
        elif isinstance(output, list):
            return ' '.join([str(item) for item in output if item])
        else:
            return str(output)

    def extract(self, image_path: str) -> Tuple[str, float]:
        for attempt in range(Config.MAX_RETRIES):
            try:
                future = self.executor.submit(self._call_replicate, image_path)
                text = future.result(timeout=Config.TIMEOUT)

                confidence = 0.95 if len(text.strip()) > 50 else 0.70
                return text, confidence

            except FuturesTimeoutError:
                logger.error(f"Timeout on attempt {attempt + 1}")
                if attempt < Config.MAX_RETRIES - 1:
                    time.sleep(Config.RETRY_DELAY)
                else:
                    return "", 0.0

            except Exception as e:
                logger.error(f"Error on attempt {attempt + 1}: {e}")
                if attempt < Config.MAX_RETRIES - 1:
                    time.sleep(Config.RETRY_DELAY)
                else:
                    return "", 0.0

        return "", 0.0

    def process_page(self, pdf_path: str, page_num: int) -> PageData:
        start = time.time()
        self.stats['attempts'] += 1

        images = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1, dpi=Config.IMAGE_DPI)
        if not images:
            raise ValueError(f"Cannot convert page {page_num}")

        image = images[0]
        img_path = f"/tmp/page_{page_num}_{int(time.time())}.png"
        image.save(img_path)

        text, confidence = self.extract(img_path)

        if confidence > 0.7:
            self.stats['successes'] += 1
        self.stats['total_time'] += time.time() - start

        return PageData(
            page_num=page_num + 1,
            text=text,
            confidence=confidence,
            image_path=img_path
        )

class TextChunker:
    @staticmethod
    def chunk_text(text: str) -> List[Tuple[str, int, int]]:
        text = re.sub(r'\s+', ' ', text).strip()
        if not text:
            return []

        sentences = re.split(r'([.!?]\s+)', text)
        full_sentences = []
        for i in range(0, len(sentences) - 1, 2):
            full_sentences.append(sentences[i] + (sentences[i+1] if i+1 < len(sentences) else ''))
        if len(sentences) % 2 == 1:
            full_sentences.append(sentences[-1])

        chunks = []
        current = ""
        start_pos = 0
        char_pos = 0

        for sentence in full_sentences:
            if len(current) + len(sentence) > Config.CHUNK_SIZE and current:
                chunks.append((current.strip(), start_pos, start_pos + len(current)))
                overlap = current[-Config.CHUNK_OVERLAP:] if len(current) > Config.CHUNK_OVERLAP else current
                current = overlap + sentence
                start_pos = char_pos - len(overlap)
            else:
                if not current:
                    start_pos = char_pos
                current += sentence
            char_pos += len(sentence)

        if current.strip():
            chunks.append((current.strip(), start_pos, char_pos))

        return chunks

class NaturalLanguageFormatter:
    @staticmethod
    def format_answer(query: str, retrieved: List[Tuple[DocumentChunk, float]]) -> str:
        if not retrieved:
            return "I couldn't find any relevant information in the uploaded documents to answer your question."

        top_chunk = retrieved[0][0]
        query_lower = query.lower()

        # For specific question patterns, extract precise answers
        if 'working days' in query_lower or ('how many' in query_lower and 'days' in query_lower):
            match = re.search(r'(?:Working Days|working days)\s*:?\s*(\d+)', top_chunk.text, re.IGNORECASE)
            if match:
                return f"There are {match.group(1)} working days.\n\nThis information was found on page {top_chunk.page_num} with {top_chunk.confidence:.0%} extraction confidence."

        if 'points' in query_lower and 'financed' in query_lower:
            match = re.search(r'(\d+)\s+[Pp]oints?\s+(?:may be|can be)?\s*financed', top_chunk.text, re.IGNORECASE)
            if match:
                return f"{match.group(1)} points may be financed.\n\nThis information was found on page {top_chunk.page_num} with {top_chunk.confidence:.0%} extraction confidence."

        if ('loan amount' in query_lower) or ('maximum loan' in query_lower):
            match = re.search(r'(?:Total Loan Amount|Loan Amount|Maximum Loan Amount)\s*:?\s*\$?\s*([\d,]+)', top_chunk.text, re.IGNORECASE)
            if match:
                amount = match.group(1)
                return f"The loan amount is ${amount}.\n\nThis information was found on page {top_chunk.page_num} with {top_chunk.confidence:.0%} extraction confidence."

        if 'interest rate' in query_lower or ('rate' in query_lower and 'interest' in query_lower):
            match = re.search(r'(?:Interest Rate|interest rate)\s*:?\s*(\d+\.?\d*\s*%)', top_chunk.text, re.IGNORECASE)
            if match:
                return f"The interest rate is {match.group(1)}.\n\nThis information was found on page {top_chunk.page_num} with {top_chunk.confidence:.0%} extraction confidence."

        if 'net pay' in query_lower:
            match = re.search(r'Net Pay\s*:?\s*(\d+)', top_chunk.text, re.IGNORECASE)
            if match:
                return f"The net pay is ${match.group(1)}.\n\nThis information was found on page {top_chunk.page_num} with {top_chunk.confidence:.0%} extraction confidence."

        if 'compensation' in query_lower and 'broker' in query_lower:
            match = re.search(r"Broker'?s?\s+Maximum\s+Compensation\s+(\d+\.?\d*%)", top_chunk.text, re.IGNORECASE)
            if match:
                return f"The broker's maximum compensation is {match.group(1)}.\n\nThis information was found on page {top_chunk.page_num} with {top_chunk.confidence:.0%} extraction confidence."

        if 'purchase price' in query_lower or ('price' in query_lower and 'purchase' in query_lower):
            match = re.search(r'Purchase Price[^\d]*\$?([\d,]+\.?\d*)', top_chunk.text, re.IGNORECASE)
            if match:
                return f"The purchase price is ${match.group(1)}.\n\nThis information was found on page {top_chunk.page_num} with {top_chunk.confidence:.0%} extraction confidence."

        if 'underwriting fee' in query_lower:
            match = re.search(r'Underwriting Fee[^\d]*\$?([\d,]+\.?\d*)', top_chunk.text, re.IGNORECASE)
            if match:
                return f"The underwriting fee is ${match.group(1)}.\n\nThis information was found on page {top_chunk.page_num} with {top_chunk.confidence:.0%} extraction confidence."

        if 'probation' in query_lower and ('period' in query_lower or 'months' in query_lower):
            match = re.search(r'(?:initial|probationary).*?(\d+).*?(?:month|months)', top_chunk.text, re.IGNORECASE)
            if match:
                return f"The probation period is {match.group(1)} months.\n\nThis information was found on page {top_chunk.page_num} with {top_chunk.confidence:.0%} extraction confidence."

        # Default: return most relevant chunk text
        answer_text = top_chunk.text[:350].strip()
        if len(top_chunk.text) > 350:
            answer_text += "..."

        return f"Based on the document: {answer_text}\n\nThis information was found on page {top_chunk.page_num} with {top_chunk.confidence:.0%} extraction confidence."

class RAGSystem:
    def __init__(self):
        self.ocr = ChandraReplicateOCR()
        self.embedding_model = embedding_model
        self.reranker = reranker_model
        self.formatter = NaturalLanguageFormatter()
        self.chunks: List[DocumentChunk] = []
        self.pages: List[PageData] = []
        self.index: Optional[faiss.IndexFlatL2] = None

    def process_document(self, pdf_path: str, progress=gr.Progress()) -> Dict:
        start = time.time()

        doc = fitz.open(pdf_path)
        total_pages = len(doc)
        doc.close()

        self.pages = []
        progress(0, desc=f"Processing {total_pages} pages")

        for i in range(total_pages):
            progress((i + 1) / total_pages * 0.7, desc=f"Page {i+1}/{total_pages}")
            try:
                page_data = self.ocr.process_page(pdf_path, i)
                self.pages.append(page_data)
            except Exception as e:
                logger.error(f"Failed page {i+1}: {e}")
                continue

        if not self.pages:
            return {'success': False, 'error': 'Failed to process any pages'}

        progress(0.75, desc="Creating chunks")
        doc_id = f"doc_{int(time.time())}"
        chunks = []
        chunk_id = 0

        for page in self.pages:
            if not page.text or len(page.text.strip()) < 50:
                continue

            page_chunks = TextChunker.chunk_text(page.text)
            for text, start_pos, end_pos in page_chunks:
                chunks.append(DocumentChunk(
                    text=text,
                    doc_id=doc_id,
                    chunk_id=chunk_id,
                    page_num=page.page_num,
                    confidence=page.confidence,
                    entities=RegexPatterns.extract_all(text)
                ))
                chunk_id += 1

        progress(0.80, desc="Generating embeddings")
        if chunks:
            texts = [c.text for c in chunks]
            batch_size = 32
            all_embeddings = []
            for i in range(0, len(texts), batch_size):
                batch = texts[i:i+batch_size]
                batch_embeddings = self.embedding_model.encode(batch)
                all_embeddings.extend(batch_embeddings)
                progress(0.80 + (i / len(texts)) * 0.15, desc=f"Embeddings {min(i+batch_size, len(texts))}/{len(texts)}")

            for chunk, emb in zip(chunks, all_embeddings):
                chunk.embedding = emb
            self.chunks.extend(chunks)
            self._rebuild_index()

        elapsed = time.time() - start
        progress(1.0, desc="Complete")

        return {
            'success': True,
            'filename': Path(pdf_path).name,
            'pages': total_pages,
            'chunks': len(chunks),
            'time': elapsed,
            'avg_confidence': np.mean([p.confidence for p in self.pages]) if self.pages else 0.0
        }

    def _rebuild_index(self):
        if not self.chunks:
            return
        embeddings = np.array([c.embedding for c in self.chunks]).astype('float32')
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings)

    def retrieve(self, query: str) -> List[Tuple[DocumentChunk, float]]:
        if not self.chunks or not self.index:
            return []

        query_emb = self.embedding_model.encode([query])[0].astype('float32')
        distances, indices = self.index.search(np.array([query_emb]), Config.RETRIEVAL_K)

        candidates = [(self.chunks[idx], 1 / (1 + dist))
                     for dist, idx in zip(distances[0], indices[0]) if idx < len(self.chunks)]

        if len(candidates) > Config.RERANK_TOP_K:
            texts = [c.text for c, _ in candidates]
            pairs = [[query, text] for text in texts]
            scores = self.reranker.compute_score(pairs)
            reranked = [(candidates[i][0], float(scores[i])) for i in range(len(candidates))]
            reranked.sort(key=lambda x: x[1], reverse=True)
            return reranked[:Config.RERANK_TOP_K]

        return candidates

    def answer_query(self, query: str) -> str:
        retrieved = self.retrieve(query)
        return self.formatter.format_answer(query, retrieved)

rag_system = RAGSystem()
print("✓ System ready")

def process_files(files, progress=gr.Progress()):
    if not files:
        return "No files uploaded.", gr.update(interactive=False)

    results = []
    for file in files:
        result = rag_system.process_document(file.name, progress)
        if result.get('success'):
            summary = f"**{result['filename']}** processed\n"
            summary += f"• {result['pages']} pages, {result['chunks']} chunks\n"
            summary += f"• Time: {result['time']:.1f}s\n"
            summary += f"• Quality: {result['avg_confidence']:.0%}\n"
            results.append(summary)
        else:
            results.append(f"Failed: {result.get('error')}")

    return "\n".join(results) + "\n\nReady for questions.", gr.update(interactive=True)

def chat_fn(message, history):
    if not message.strip():
        return history

    if not rag_system.chunks:
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": "Upload documents first."})
        return history

    answer = rag_system.answer_query(message)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": answer})
    return history

custom_css = """
* {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif !important;
}
.gradio-container {
    background: #ffffff !important;
    max-width: 1400px !important;
}
#header {
    background: white;
    padding: 24px;
    border-bottom: 1px solid #e0e0e0;
}
.title {
    font-size: 28px;
    font-weight: 600;
    color: #000000;
    margin: 0;
}
.subtitle {
    font-size: 14px;
    color: #000000;
    margin-top: 6px;
}
#chatbot {
    background: #ffffff !important;
    border: 1px solid #d0d0d0 !important;
    border-radius: 8px !important;
}
#chatbot, #chatbot *, #chatbot > div, #chatbot .message-wrap, #chatbot .wrap {
    background: #ffffff !important;
}
.message, .message *, .bot, .bot *, .user, .user * {
    color: #000000 !important;
}
.user {
    background: #f5f5f5 !important;
    border-left: 3px solid #ff6b35 !important;
}
.bot {
    background: #ffffff !important;
}
#status {
    background: white;
    padding: 16px;
    border-radius: 8px;
    border-left: 3px solid #ff6b35;
    color: #000000;
}
.primary-btn {
    background: linear-gradient(135deg, #ff6b35 0%, #ff8956 100%) !important;
    color: white !important;
}
#msg, #msg * {
    background: white !important;
    color: #000000 !important;
}
.secondary-btn {
    background: white !important;
    color: #000000 !important;
}
label, h1, h2, h3, h4, h5, h6, p, span, div, b, strong {
    color: #000000 !important;
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="orange")) as demo:
    with gr.Row(elem_id="header"):
        with gr.Column():
            gr.Markdown("# Document Intelligence System", elem_classes="title")
            gr.Markdown("Chandra OCR • BGE Semantic Search • A100 Accelerated", elem_classes="subtitle")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Upload Documents")
            files = gr.File(label="PDF Files", file_count="multiple", file_types=[".pdf"], height=120)
            process_btn = gr.Button("Process Documents", variant="primary", elem_classes="primary-btn", size="lg")
            status = gr.Markdown("Ready", elem_id="status")

        with gr.Column(scale=2):
            gr.Markdown("### Ask Questions")
            chatbot = gr.Chatbot(height=550, elem_id="chatbot", show_copy_button=True, type="messages", label="")

            with gr.Row():
                msg = gr.Textbox(label="Your Question", placeholder="Type your question...", scale=5, lines=2, elem_id="msg")
                send = gr.Button("Send", variant="primary", scale=1, elem_classes="primary-btn")

            gr.Markdown("**Quick Questions:**")
            with gr.Row():
                gr.Button("What is the maximum loan amount?", size="sm", elem_classes="secondary-btn").click(lambda: "What is the maximum loan amount?", outputs=msg)
                gr.Button("What is the broker's maximum compensation?", size="sm", elem_classes="secondary-btn").click(lambda: "What is the broker's maximum compensation?", outputs=msg)
                gr.Button("How many points can be financed?", size="sm", elem_classes="secondary-btn").click(lambda: "How many points can be financed?", outputs=msg)

            clear = gr.Button("Clear", variant="secondary", elem_classes="secondary-btn")

    process_btn.click(process_files, inputs=files, outputs=[status, send])
    send.click(chat_fn, inputs=[msg, chatbot], outputs=chatbot).then(lambda: "", outputs=msg)
    msg.submit(chat_fn, inputs=[msg, chatbot], outputs=chatbot).then(lambda: "", outputs=msg)
    clear.click(lambda: [], outputs=chatbot)

print("\n" + "=" * 80)
print("SYSTEM READY - A100 GPU")
print("=" * 80)

demo.launch(share=True, debug=False, server_port=7861)

INITIALIZING SYSTEM - A100 OPTIMIZED

[1/4] Installing dependencies...
✓ Dependencies installed

[2/4] Importing libraries...
✓ Libraries imported
Device: cuda (NVIDIA A100-SXM4-40GB)

[3/4] Loading BGE embeddings...


BGE: 100%|██████████| 100/100


✓ BGE loaded

[4/4] Building system...
✓ System ready

SYSTEM READY - A100 GPU
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f2353f7bb18c02c48d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
